# ETI Research Core v0.2 — budget-aware CPU screening
Run only when intentionally starting a new experiment. This notebook uses public market data and never places orders.

## User-set Kaggle spending ceilings
- Daily ceiling: USD 10
- Monthly ceiling: USD 100
- These are hard limits, not targets. Check Kaggle's own usage/billing display before running; this notebook cannot read or enforce dollar billing.
- CPU only. One symbol. No hyperparameter sweep. Estimated model-fit calls are capped before training.

## Actual model export
- The run performs nested evaluation, then fits one deployable BTC/USDT candidate using a chronological 80/20 fit/calibration split.
- The candidate is saved as a content-addressed `.joblib` artifact under `artifacts/models/`, with SHA-256 and feature-schema metadata in the JSON report.
- This is a research candidate, not approval for live trading.


In [ ]:
from pathlib import Path
import subprocess, sys
repo = Path('/kaggle/working/eti-research-core')
if not repo.exists():
    subprocess.run(['git', 'clone', '--branch', 'main', 'https://github.com/hamza00alik-source/eti-research-core.git', str(repo)], check=True)
else:
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only'], check=True)
%cd /kaggle/working/eti-research-core
%pip install -r requirements.txt


In [ ]:
# First-stage screening: intentionally small and CPU-only.
EXCHANGE = 'binance'
SYMBOL = 'BTC/USDT'
TIMEFRAME = '1h'
CANDLE_LIMIT = 2000
HORIZON = 6
MIN_TRAIN = 400
VALIDATION_SIZE = 100
TEST_SIZE = 100
STEP_SIZE = 200
MAX_MODEL_FITS = 32
OUTPUT = 'artifacts/eti_budget_screen.json'
MODEL_DIR = 'artifacts/models'
os.environ['ETI_ARTIFACT_DIR'] = MODEL_DIR

command = [
    sys.executable, '-m', 'eti.pipeline',
    '--exchange', EXCHANGE, '--symbol', SYMBOL,
    '--timeframe', TIMEFRAME, '--limit', str(CANDLE_LIMIT),
    '--horizon', str(HORIZON), '--min-train', str(MIN_TRAIN),
    '--validation-size', str(VALIDATION_SIZE),
    '--test-size', str(TEST_SIZE), '--step-size', str(STEP_SIZE),
    '--max-model-fits', str(MAX_MODEL_FITS),
    '--evaluation', 'nested', '--output', OUTPUT,
    '--save-model', '--model-id', 'eti-btc-1h', '--model-version', '0.1.0',
]
print('Starting one BTC/USDT CPU screening run; dollar spend must be checked in Kaggle UI.')
subprocess.run(command, check=True)
print(Path(OUTPUT).read_text())
print('Model artifact files:')
for path in sorted(Path(MODEL_DIR).rglob('*.joblib')):
    print(path, path.stat().st_size, 'bytes')
